In [47]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns
import os
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBRegressor
from sklearn.preprocessing import StandardScaler


In [48]:
input_path = f'/home/isha/Desktop/Projects/03_Remaining_Useful_Life_Prediction/data/raw/'
output_path = f'/home/isha/Desktop/Projects/03_Remaining_Useful_Life_Prediction/data/artifacts/'

In [49]:
train_df = pd.read_csv(os.path.join(input_path, 'train_FD001.txt'), sep=r"\s+", header=None)
columns = ( ["engine_id", "cycle"] + [f"operational_setting_{i}" for i in range(1, 4)] + 

           [f"s{i}" for i in range(1, 22)])
train_df.columns = columns


In [50]:
RUL_CAP = 125
max_cycles = (train_df.groupby('engine_id')['cycle'].max().rename('max_cycle'))

train_df = train_df.join(max_cycles, on='engine_id') 
train_df['RUL'] = train_df['max_cycle'] - train_df['cycle']
train_df['RUL'] = train_df['RUL'].clip(upper = RUL_CAP)       #Cap RUL at RUL_CAP
train_df.drop(columns=['max_cycle'], inplace=True)
train_df.head()

,engine_id,cycle,operational_setting_1,operational_setting_2,operational_setting_3,s1,s2,s3,s4,s5,...,s13,s14,s15,s16,s17,s18,s19,s20,s21,RUL
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190,125
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236,125
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442,125
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739,125
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044,125


In [51]:
test_df = pd.read_csv(os.path.join(input_path, 'test_FD001.txt'), sep=r"\s+", header=None)
test_df.columns = columns
test_df.head()

,engine_id,cycle,operational_setting_1,operational_setting_2,operational_setting_3,s1,s2,s3,s4,s5,...,s12,s13,s14,s15,s16,s17,s18,s19,s20,s21
0,1,1,0.0023,0.0003,100.0,518.67,643.02,1585.29,1398.21,14.62,...,521.72,2388.03,8125.55,8.4052,0.03,392,2388,100.0,38.86,23.3735
1,1,2,-0.0027,-0.0003,100.0,518.67,641.71,1588.45,1395.42,14.62,...,522.16,2388.06,8139.62,8.3803,0.03,393,2388,100.0,39.02,23.3916
2,1,3,0.0003,0.0001,100.0,518.67,642.46,1586.94,1401.34,14.62,...,521.97,2388.03,8130.10,8.4441,0.03,393,2388,100.0,39.08,23.4166
3,1,4,0.0042,0.0000,100.0,518.67,642.44,1584.12,1406.42,14.62,...,521.38,2388.05,8132.90,8.3917,0.03,391,2388,100.0,39.00,23.3737
4,1,5,0.0014,0.0000,100.0,518.67,642.51,1587.19,1401.92,14.62,...,522.15,2388.03,8129.54,8.4031,0.03,390,2388,100.0,38.99,23.4130


In [52]:
y_test = pd.read_csv((os.path.join(input_path, 'RUL_FD001.txt')),  sep=r"\s+", header=None)
y_test.columns = ['RUL']
y_test.head()

,RUL
0,112
1,98
2,69
3,82
4,91


In [53]:
sensor_col = [f"s{i}" for i in range(1,22)]

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(train_df[sensor_col]))
y_train = train_df["RUL"]   #train_df.groupby('engine_id')['RUL'].apply(list)
y_test = y_test
X_test_scaled = pd.DataFrame(scaler.transform(test_df[sensor_col]))

In [54]:
os.makedirs("artifacts", exist_ok=True)

X_train_scaled.to_csv("artifacts/X_train_scaled.csv", index=False)
X_test_scaled.to_csv("artifacts/X_test_scaled.csv", index=False)
#y_train.to_csv("artifacts/y_train.csv", index=False )


In [55]:
#For calibration
CALIBRATION_FRACTION = 0.20
RANDOM_SEED = 42

engine_ids = train_df["engine_id"].values
unique_engines = np.unique(engine_ids)
n_engines = len(unique_engines)

print(f"Total engines = {n_engines}")

Total engines = 100


In [56]:
rng = np.random.default_rng(RANDOM_SEED)
shuffled_engines = rng.permutation(unique_engines)

n_cal = max(1, int(n_engines * CALIBRATION_FRACTION))
calibration_engines = shuffled_engines[:n_cal]
training_engines = shuffled_engines[n_cal:]

print("Training engines:", len(training_engines))
print("Calibration engines:", len(calibration_engines))

Training engines: 80
Calibration engines: 20


In [57]:
calibration_mask = np.isin(engine_ids, calibration_engines)
training_mask = ~calibration_mask

In [58]:
training_mask, calibration_mask

(array([ True,  True,  True, ..., False, False, False], shape=(20631,)),
 array([False, False, False, ...,  True,  True,  True], shape=(20631,)))

In [59]:
X_train_processed = X_train_scaled.copy()
y_train_processed = y_train.copy()

In [60]:
X_train_proper = (X_train_processed.loc[training_mask].reset_index(drop=True))
y_train_proper = (y_train_processed.loc[training_mask].reset_index(drop=True))
X_cal = (X_train_processed.loc[calibration_mask].reset_index(drop=True))
y_cal = (y_train_processed.loc[calibration_mask].reset_index(drop=True))

X_train_proper.columns = sensor_col
X_cal.columns = sensor_col

In [61]:
print(X_train_proper.shape)
print(y_train_proper.shape)

print(X_cal.shape)
print(y_cal.shape)

(16779, 21)
(16779,)
(3852, 21)
(3852,)


print(f'X_train: {X_train_scaled.shape}')
print(f'y_train: {y_train.shape} | range {y_train.min():.0f} - {y_train.max():.0f}')
print(f'X_test: {X_test_scaled.shape}')
print(f'y_test: {y_test.shape} ')

In [62]:
def evaluate_model(true, predicted):
    mae = mean_absolute_error(true, predicted)
    rmse = np.sqrt(mean_squared_error(true, predicted))
    r2_square = r2_score(true, predicted)
    return mae, rmse, r2_square

In [63]:
xgboost_model = XGBRegressor(n_estimators = 32, learning_rate = 0.1, random_state=42)
xgboost_model_fit = xgboost_model.fit(X_train_proper, y_train_proper) # Train model    

In [64]:
#INCOMPLETE FROM HERE

ALPHA = 0.10
NOMINAL = 1 - ALPHA

cal_predictions = xgboost_model.predict(X_cal)

residuals = np.abs(
    y_cal.values -
    cal_predictions
)

n_cal = len(y_cal)

level = min(
    np.ceil((1-ALPHA)*(n_cal+1))/n_cal,
    1.0
)

q_hat = np.quantile(
    residuals,
    level
)

print("Calibration size :", n_cal)
print("Quantile level :", level)
print("q_hat :", q_hat)

Calibration size : 3852
Quantile level : 0.9003115264797508
q_hat : 31.634688998308494


In [65]:
test_last = (
    test_df
    .groupby("engine_id")
    .tail(1)
)

X_test = pd.DataFrame(
    scaler.transform(
        test_last[sensor_col]
    ),
    columns=sensor_col
)

y_test = y_test["RUL"].values

In [66]:
#Prediction
y_hat = xgboost_model.predict(X_test)

lower = y_hat - q_hat
upper = y_hat + q_hat

In [67]:
covered = (
    (y_test >= lower) &
    (y_test <= upper)
)

coverage = covered.mean()

avg_width = np.mean(
    upper - lower
)

print(f"Coverage : {coverage:.3f}")
print(f"Nominal : {NOMINAL:.3f}")
print(f"Average width : {avg_width:.3f}")

Coverage : 0.900
Nominal : 0.900
Average width : 63.269


In [68]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

mae = mean_absolute_error(
    y_test,
    y_hat
)

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_hat
    )
)

r2 = r2_score(
    y_test,
    y_hat
)

print(f"MAE : {mae:.3f}")
print(f"RMSE : {rmse:.3f}")
print(f"R² : {r2:.3f}")

MAE : 13.435
RMSE : 18.348
R² : 0.805


In [69]:
results = [{
    "model": "XGBoost",
    "q_hat": q_hat,
    "coverage": coverage,
    "avg_width": avg_width,
    "_y_hat": y_hat,
    "_lower": lower,
    "_upper": upper,
}]

res_df = pd.DataFrame(results)

res_df["_valid"] = res_df["coverage"] >= NOMINAL

display(
    res_df[
        ["model", "q_hat", "coverage", "avg_width", "_valid"]
    ].rename(
        columns={
            "_valid": "meets_coverage_guarantee"
        }
    )
)

,model,q_hat,coverage,avg_width,meets_coverage_guarantee
0,XGBoost,31.634689,0.9,63.269378,True
